# 06 Cross-Validation (M2)(Faleyimu Olaoluwa Halleluyah)

This notebook checks how well the linear regression and random forest models work using 5-fold cross-validation, so the results do not depend on one lucky split.

## 1. Load the cleaned data

We load the cleaned file from M1 (1,337 rows, 7 columns).

In [2]:
import pandas as pd

# loading the cleaned data
df = pd.read_csv('../data/insurance_clean.csv')

print(df.shape)
df.head()

(1337, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## 2. Prepare the data

We separate the answer we want to predict (charges) from the clues (all other columns). The text columns (sex, smoker, region) are turned into numbers, the same way M7 did it, so our results match the group's.

In [3]:
# clues (X) and the answer we want to predict (y)
X = pd.get_dummies(df.drop(columns=['charges']), drop_first=True)
y = df['charges']

print(X.shape)
X.head()

(1337, 8)


,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest
0,19,27.900,0,False,True,False,False,True
1,18,33.770,1,True,False,False,True,False
2,28,33.000,3,True,False,False,True,False
3,33,22.705,0,True,False,True,False,False
4,32,28.880,0,True,False,True,False,False


## 3. Cross-validation: Linear Regression

We split the data into 5 parts. The model trains on 4 parts and is tested on the 5th, and this repeats 5 times so every part gets a turn as the test set. We measure MAE (average error in charges, lower is better) and R2 (how much of the pattern the model explains, closer to 1 is better).

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

# the linear regression model
lin_model = LinearRegression()

# run 5-fold cross-validation and score by R2
r2_scores = cross_val_score(lin_model, X, y, cv=5, scoring='r2')

print(r2_scores)
print("Average R2:", r2_scores.mean())

[0.76148609 0.70651382 0.77741544 0.73269494 0.7555821 ]
Average R2: 0.7467384760760104


**Linear regression MAE.** MAE is the average size of the mistake in the predicted charges, so lower is better. scikit-learn reports error scores as negative numbers, so we flip the sign to get the real MAE.

In [5]:
# same cross-validation, but scored by MAE
mae_scores = -cross_val_score(lin_model, X, y, cv=5, scoring='neg_mean_absolute_error')

print(mae_scores)
print("Average MAE:", mae_scores.mean())

[4240.03525568 4260.92138308 4024.58641942 4256.72515009 4216.75412718]
Average MAE: 4199.80446709229


## 4. Cross-validation: Random Forest

Same 5-fold test, now with the random forest model. A random forest is many decision trees voting together. We use random_state=42 so the result is the same every time we run it.

In [7]:
from sklearn.ensemble import RandomForestRegressor

# the random forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

# run 5-fold cross-validation and score by R2
rf_r2_scores = cross_val_score(rf_model, X, y, cv=5, scoring='r2')

print(rf_r2_scores)
print("Average R2:", rf_r2_scores.mean())

[0.85389188 0.77901285 0.86917054 0.82767286 0.85105358]
Average R2: 0.8361603445279624


**Random forest MAE.** Same idea as before: the average size of the mistake in the predicted charges, so lower is better.

In [8]:
# same cross-validation, but scored by MAE
rf_mae_scores = -cross_val_score(rf_model, X, y, cv=5, scoring='neg_mean_absolute_error')

print(rf_mae_scores)
print("Average MAE:", rf_mae_scores.mean())

[2684.27901216 2920.23330017 2539.47367352 2662.27304007 2754.97885199]
Average MAE: 2712.2475755812757


## 5. Comparing the two models

Both models were tested with the same 5-fold cross-validation. For R2, higher is better. For MAE, lower is better.

In [9]:
# put the average scores of both models in one table
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'Average R2': [r2_scores.mean(), rf_r2_scores.mean()],
    'Average MAE': [mae_scores.mean(), rf_mae_scores.mean()]
})

# round so it is easy to read
results = results.round(2)
results

,Model,Average R2,Average MAE
0,Linear Regression,0.75,4199.80
1,Random Forest,0.84,2712.25


## 6. Conclusion

The random forest did better than linear regression on both measures. It explains about 84% of the variation in charges (R2 of 0.84) compared with 75% for linear regression, and its average mistake is about 2,712 instead of about 4,200. The scores stayed fairly steady across the 5 folds for both models, and the random forest was better in every fold. for both models, so the result does not depend on one lucky split. We recommend the random forest as the better model for predicting insurance charges.